# GeoLibre exploration — Sedona gold geo tables

Analyst workbench for OpenDesk geospatial analytics (SPEC-W8 Part B2).

**Flow:** PostGIS (`contact_locations`, `service_areas` in booking-service) →
extracts on MinIO (`s3://lake/extracts/contact_locations/`,
`s3://lake/extracts/service_areas/`) → **Apache Sedona** Spark job
(`infra/lakehouse/spark/jobs/geo_analytics.py`) → **Iceberg gold tables**
(`gold.geo_demand_h3`, `gold.geo_service_area_coverage`,
`gold.geo_hourly_density`, Trino-visible) → **GeoLibre** map rendering
here / in the deployed workbench on `:8085` (`/gis/*` via APISIX).

This notebook pulls a sample of `gold.geo_demand_h3` (H3 res-8 demand
cells per tenant per day, cell polygon as WKT) plus the service-areas
extract, renders both on a GeoLibre `Map`, then saves/loads a
`.geolibre.json` project.


In [ ]:
# GeoLibre + a Trino client for pulling gold-table extracts.
%pip install -q geolibre trino shapely


## 1. Extract a sample of `gold.geo_demand_h3` from Trino

Trino converts the WKT cell polygons straight to GeoJSON geometries, so
the extract is a ready-to-render GeoJSON FeatureCollection. Adjust
`TENANT_ID` / `DAY` to your dataset (demo seeds cluster around Lagos,
`6.5244, 3.3792`).


In [ ]:
import json
import trino

TRINO_HOST = 'localhost'   # 'trino' inside the opendesk docker network
TRINO_PORT = 8088          # host port mapping; 8080 in-network
TENANT_ID = 'demo-tenant'  # TODO: replace with a real tenant id

conn = trino.dbapi.connect(host=TRINO_HOST, port=TRINO_PORT, user='analyst', catalog='iceberg', schema='gold')
cur = conn.cursor()
cur.execute(
    '''
    SELECT h3_cell_str, bookings,
           ST_AsGeoJSON(ST_GeometryFromText(cell_wkt)) AS geometry
    FROM iceberg.gold.geo_demand_h3
    WHERE tenant_id = ?
    ORDER BY day DESC, bookings DESC
    LIMIT 500
    ''',
    (TENANT_ID,),
)
demand_geojson = {
    'type': 'FeatureCollection',
    'features': [
        {
            'type': 'Feature',
            'geometry': json.loads(geom),
            'properties': {'h3_cell': cell, 'bookings': int(bookings)},
        }
        for cell, bookings, geom in cur.fetchall()
    ],
}
print(f"{len(demand_geojson['features'])} H3 cells extracted")


## 2. Service areas

The same service-areas extract the Sedona job reads
(`s3://lake/extracts/service_areas/` — GeoJSON FeatureCollection whose
features carry `{tenant_id, service_area_id, name}` properties). Point
`SERVICE_AREAS_GEOJSON` at a downloaded copy, or keep the small inline
Lagos demo polygons below.


In [ ]:
SERVICE_AREAS_GEOJSON = None  # e.g. '/data/service_areas.geojson'

if SERVICE_AREAS_GEOJSON:
    with open(SERVICE_AREAS_GEOJSON) as fh:
        service_areas_geojson = json.load(fh)
else:
    # Inline demo: two Lagos service areas (Victoria Island + Ikeja).
    service_areas_geojson = {
        'type': 'FeatureCollection',
        'features': [
            {
                'type': 'Feature',
                'properties': {'tenant_id': 'demo-tenant', 'service_area_id': 'demo-vi', 'name': 'Victoria Island'},
                'geometry': {'type': 'Polygon', 'coordinates': [[
                    [3.39, 6.41], [3.45, 6.41], [3.45, 6.45], [3.39, 6.45], [3.39, 6.41],
                ]]},
            },
            {
                'type': 'Feature',
                'properties': {'tenant_id': 'demo-tenant', 'service_area_id': 'demo-ikeja', 'name': 'Ikeja'},
                'geometry': {'type': 'Polygon', 'coordinates': [[
                    [3.32, 6.57], [3.38, 6.57], [3.38, 6.62], [3.32, 6.62], [3.32, 6.57],
                ]]},
            },
        ],
    }


## 3. Render on a GeoLibre map

Leafmap-style API: `Map(center=(lat, lng), zoom=...)`, then
`add_geojson(...)` per layer. Centered on Lagos (6.5244, 3.3792).


In [ ]:
from geolibre import Map

m = Map(center=(6.5244, 3.3792), zoom=11)
m.add_geojson(
    demand_geojson,
    layer_name='gold.geo_demand_h3 (bookings per H3 cell/day)',
    style={'fillColor': '#e07a5f', 'fillOpacity': 0.5, 'color': '#9c4a32', 'weight': 1},
)
m.add_geojson(
    service_areas_geojson,
    layer_name='service areas',
    style={'fillColor': '#3d405b', 'fillOpacity': 0.1, 'color': '#3d405b', 'weight': 2},
)
m


## 4. Save / reload a `.geolibre.json` project

GeoLibre projects persist layers + view state as JSON — share them with
other analysts or reopen them in the deployed workbench
(`http://localhost:8085`, or `/gis/` through the gateway).


In [ ]:
m.save('opendesk-geo-demo.geolibre.json')

with open('opendesk-geo-demo.geolibre.json') as fh:
    project = json.load(fh)
print('saved project keys:', sorted(project)[:5], '...')

# Reload later (new kernel / the deployed workbench):
# from geolibre import Map
# m2 = Map.load('opendesk-geo-demo.geolibre.json')


## Notes

- **Cell geometry** lives in `gold.geo_demand_h3.cell_wkt` (WKT); Trino
  turns it into geometries with `ST_GeometryFromText`. `gold.geo_hourly_density`
  omits geometry — join on `h3_cell` for staffing heatmaps.
- **H3**: cells are H3 res-8 (Sedona `ST_H3CellIDs`; see the job docstring
  for the H3-vs-geohash decision).
- **Embedding**: the deployed GeoLibre supports a chrome-less `maponly`
  mode (`/gis/?mode=maponly`) for admin-dashboard iframes — see
  `docs/geospatial-infra.md`.
